In [4]:
!pip install -U peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.5 MB/s eta 0:00:00:00:0100:01


In [1]:
import os
import torch
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, GenerationConfig
from peft import PeftModel
from huggingface_hub import login

drive.mount('/content/drive')
base_path = "/content/drive/MyDrive/Colab Notebooks/Quant"
os.chdir(base_path)

from dotenv import load_dotenv
load_dotenv(override=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


True

In [2]:
hf_token = os.environ.get("HUGGINGFACE_KEY")
login(token=hf_token)

In [3]:
base_model_id = "google/gemma-2b"
new_model_path = "gemma-finetuned-final"

tokenizer = AutoTokenizer.from_pretrained(new_model_path)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, new_model_path)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GemmaForCausalLM(
      (model): GemmaModel(
        (embed_tokens): Embedding(256000, 2048, padding_idx=0)
        (layers): ModuleList(
          (0-17): 18 x GemmaDecoderLayer(
            (self_attn): GemmaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
            

In [30]:
template = "<start_of_turn>user\n{instruction}\n\nInput: {input_str}<end_of_turn>\n decision: "
instruction = ("You are a financial decision assistant.\n\n"
"Your task is to decide whether to BUY, HOLD, or SELL a stock based on the following information:\n"
"1. LSTM model output (Estimated return for today based on stock price history of the past 60 days)\n"
"2. BERT model output (Analyzes recent 100 news article)\n"
"3. Current macroeconomic conditions\n"
"4. Current portfolio status\n\n"
""
"Rules:\n"
"- Your answer must contain ONLY:\n  "
"1) Decision: BUY / HOLD / SELL\n  "
"2) Amount of Shares to Trade\n  "
"3) Reason: one or two short sentences explaining the main factors.\n"
# "- Do NOT provide extra commentary.\n"
# "- Stop immediately after the reason."
)

lstm_output = "LSTM model output : +1.25%"
bert_output = "BERT model output : 30% Positive, 60% Neutral, 10% Negative"
macro_data = "Inflation Rate Higher than normal"
open_price = 312
cash = 30043
shares_owned = 30


input_str = f"""
LSTM Prediction: {lstm_output}
BERT Sentiment: {bert_output}
Opening Price: ${open_price}
Cash: ${cash}
Current Holdings: {shares_owned} shares
"""

prompt = template.format(
    instruction=instruction,
    input_str=input_str
)

input_ids = tokenizer(prompt, return_tensors="pt").to(model.device)
input_len = input_ids.input_ids.shape[1]

config = GenerationConfig(
    do_sample=False,
    max_new_tokens=50, 
    repetition_penalty=1.1,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id
)

outputs = model.generate(**input_ids, generation_config=config)
new_response = tokenizer.decode(outputs[0][:], skip_special_tokens=True)

print(new_response.strip())

user
You are a financial decision assistant.

Your task is to decide whether to BUY, HOLD, or SELL a stock based on the following information:
1. LSTM model output (Estimated return for today based on stock price history of the past 60 days)
2. BERT model output (Analyzes recent 100 news article)
3. Current macroeconomic conditions
4. Current portfolio status

Rules:
- Your answer must contain ONLY:
  1) Decision: BUY / HOLD / SELL
  2) Amount of Shares to Trade
  3) Reason: one or two short sentences explaining the main factors.


Input: 
LSTM Prediction: LSTM model output : +1.25%
BERT Sentiment: BERT model output : 30% Positive, 60% Neutral, 10% Negative
Opening Price: $312
Cash: $30043
Current Holdings: 30 shares

 decision: <strong>SELL</strong>
 shares: <strong>13</strong>
 reason: Negative forecast and bearish sentiment under weak macro conditions suggest reducing exposure.Więhematical model output : -0.78%
település: <strong> vicissitude</strong>
főzések:
